# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [21]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [ ]:
# Load dataset
df = pd.read_csv("../data/raw/rainfall in india 1901-2015.csv")

# Verify dataset
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.sample(5)

Dataset shape: 4116 rows x 19 columns


,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec
132,ARUNACHAL PRADESH,1938,144.8,121.6,340.5,395.3,306.5,1511.3,1355.1,790.8,877.3,220.6,59.7,5.6,6129.0,266.4,1042.3,4534.5,285.8
3294,TELANGANA,1998,5.2,22.2,15.2,15.5,27.9,149.4,264.0,290.5,209.3,79.4,14.5,0.0,1093.0,27.3,58.5,913.2,93.9
2156,WEST MADHYA PRADESH,2010,2.2,5.4,0.8,0.1,0.4,62.2,258.5,291.5,136.1,13.6,4.6,0.8,776.3,7.6,1.4,748.3,19.0
1595,HIMACHAL PRADESH,1909,111.0,82.1,25.4,31.7,19.0,107.6,292.4,234.5,105.9,17.6,0.0,49.9,1077.0,193.0,76.1,740.4,67.5
2003,EAST RAJASTHAN,1972,0.4,4.4,0.0,1.1,0.8,72.4,86.7,259.9,21.9,3.3,3.3,0.1,454.3,4.8,2.0,440.9,6.6


# Task 1: Select Data

In [31]:
# For this dataset, all columns are relevant to rainfall prediction.
# No columns are dropped at this stage.

columns_to_keep = df.columns
columns_to_drop = []

drop_reason = {
    "None": "All rainfall and year-related features contribute to prediction"
}

df_selected = df[columns_to_keep].copy()

print(f"Shape after column selection: {df_selected.shape}")
df_selected.head()

Shape after column selection: (4116, 19)


,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,332.6,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,666.2,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,339.0,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,820.4,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,297.0,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8


# Task 2: Clean Data

In [32]:
df_clean = df_selected.copy()

# Handle missing values using median (best for rainfall)
numerical_cols = df_clean.select_dtypes(include=np.number).columns
df_clean[numerical_cols] = df_clean[numerical_cols].fillna(
    df_clean[numerical_cols].median()
)

print("Missing values after imputation:")
print(df_clean.isnull().sum().sum())

Missing values after imputation:
0


## Remove duplicates

In [33]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


## Handle outliers

In [34]:
def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    dataframe[column] = dataframe[column].clip(lower, upper)
    return dataframe

for col in numerical_cols:
    df_clean = cap_outliers_iqr(df_clean, col)

print("Outliers capped using IQR method")

Outliers capped using IQR method


# Task 3: Feature Engineering

In [35]:
# Average rainfall feature
rainfall_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

df_clean['Avg_Rainfall'] = df_clean[rainfall_cols].mean(axis=1)

# Rainfall anomaly
df_clean['Rainfall_Anomaly'] = (
    df_clean['Avg_Rainfall'] - df_clean['Avg_Rainfall'].mean()
)

df_clean.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec,Avg_Rainfall,Rainfall_Anomaly
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,66.1,29.2,2.3000,229.1625,517.5,365.1,481.1000,332.6000,348.85,113.5125,33.6,2878.075,119.6,455.95,1696.3,480.2,561.075000,229.975652
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,66.1,12.2,0.0000,229.1625,537.1,228.9,710.0875,513.4125,197.20,113.5125,43.6,2878.075,119.6,455.95,2185.9,480.2,592.944444,261.845097
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,66.1,0.0,1.0000,229.1625,479.9,728.4,326.7000,339.0000,181.20,113.5125,43.6,2878.075,119.6,236.10,1874.0,480.2,556.236111,225.136764
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,120.0625,229.1625,495.1,502.0,160.1000,513.4125,222.20,113.5125,40.1,2878.075,24.1,455.95,1977.6,480.2,563.315278,232.215930
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9000,229.1625,628.7,368.7,330.5000,297.0000,260.70,25.4000,43.6,2566.700,1.3,309.70,1624.9,480.2,505.725694,174.626347


# Task 4: Integrate Scraped Temperature Data

In [36]:
# Load scraped temperature data
temp_df = pd.read_csv("../data/raw/Scraped_temperature.csv")

print("Temperature dataset shape:", temp_df.shape)
temp_df.head()

Temperature dataset shape: (26, 2)


,Year,Avg_Temperature
0,1990,25.60
1,1991,24.83
2,1992,25.78
3,1993,26.83
4,1994,24.72


## Merge

In [72]:
df_integrated = pd.merge(
    df_clean,
    temp_df,
    on="Year",
    how="left"
)

print("Integrated dataset shape:", df_integrated.shape)
df_integrated.head()

KeyError: 'Year'

# Task5 Format + Save Final Dataset

In [68]:
df_final = df_integrated.copy()

# Ensure numeric types
df_final = df_final.apply(pd.to_numeric, errors='ignore')

# Final checks
print("="*50)
print("FINAL DATASET")
print("="*50)
print("Shape:", df_final.shape)
print("Missing:", df_final.isnull().sum().sum())
print("Duplicates:", df_final.duplicated().sum())

NameError: name 'df_integrated' is not defined

In [73]:
OUTPUT_PATH = "../data/prepared_rainfall_dataset.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Prepared dataset saved to: {OUTPUT_PATH}")

NameError: name 'df_final' is not defined